# Architecture Refinements: from "it runs" to "it trains well"

> **Previous recap**: In Part 4 we built a MiniGPT using the classic Transformer recipe:
> - Residual after attention, then LayerNorm (**Post-Norm**)
> - ReLU in the FFN (**ReLU FFN**)
> - LayerNorm for normalization (**LayerNorm**)
>
> **But modern LLMs (LLaMA, GPT-4, DeepSeek) use a different recipe.**
>
> Goal for this part: **upgrade the Part-4 Transformer block step by step into the modern LLM-style block, and understand *why* each change helps.**

We will make three upgrades. For each upgrade we ask three questions:
1. What is the old thing? How is it computed?
2. What is the problem with the old thing?
3. What is the new thing? Why is it better?

**We will use tiny numeric examples and compute by hand, so every step is trackable.**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 0. Quick recap: what does the Part-4 block look like?

Let's paste the block you already know from Part 4, and modify it one change at a time.


In [ ]:
# === This is the Part 4 version: our starting point for upgrades ===

class FeedForward_Old(nn.Module):
    """Original FFN: two Linear layers with a ReLU in the middle."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock_Old(nn.Module):
    """Post-Norm + LayerNorm + ReLU FFN (Part 4 style)."""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # plain LayerNorm
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # Post-Norm: sublayer -> add residual -> then normalize
        x = self.norm1(x + self.attention(x, x, x, need_weights=False)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print("[ok] This is the Part 4 baseline. Next we upgrade it step by step.")
print("Upgrade path: LayerNorm -> RMSNorm -> ReLU -> SwiGLU -> Post-Norm -> Pre-Norm")


---

### 1. Upgrade 1: LayerNorm -> RMSNorm

#### 1.1 What does LayerNorm compute? Do it by hand with 4 numbers

Assume a token vector is `[1, 3, 5, 7]` (4 dims, matching our Part-4 `d_model=4`).

**LayerNorm** makes the mean 0 and the standard deviation 1.

A useful analogy: grading on a curve. No matter how high or low raw scores are, after normalization the class average becomes 0 and the spread becomes consistent.

Steps:

```
1. mean      mu = (x1 + x2 + x3 + x4) / 4
2. variance  sigma^2 = ((x1-mu)^2 + ... + (x4-mu)^2) / 4
3. std       sigma = sqrt(sigma^2)
4. normalize x' = (x - mu) / sigma
5. affine    y = gamma * x' + beta
```

`gamma` and `beta` are learnable parameters.

Next we compute LayerNorm for `[1, 3, 5, 7]` by hand.


In [ ]:
# === LayerNorm computed by hand ===
print("=== LayerNorm computed by hand: input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Step 1: mean
mu = x.mean()
print(f"Step 1 - mean mu = (1+3+5+7)/4 = {mu:.1f}")

# Step 2: variance (divide by N, not N-1)
var = torch.mean((x - mu) ** 2)
print("Step 2 - variance sigma^2 = ((1-4)^2+(3-4)^2+(5-4)^2+(7-4)^2)/4")
print(f"       = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Step 3: std
sigma = torch.sqrt(var)
print(f"Step 3 - std sigma = sqrt({var:.1f}) = {sigma:.4f}")

# Step 4: normalize
x_norm = (x - mu) / sigma
print(f"Step 4 - normalize: (x - 4) / {sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"       x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"       normalized: {[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"       mean: {x_norm.mean():.4f} (=0), std: {x_norm.std(unbiased=False):.4f} (=1)")

# Step 5: scale (assume gamma=[1,1,1,1], beta=[0,0,0,0])
print("Step 5 - scale: with gamma=1 and beta=0, output equals the normalized result")
print("       gamma and beta are learned during training so the model can rescale/shift")


#### 1.2 The problem with LayerNorm: it computes something you may not need

LayerNorm computes two statistics:
- **mu (mean)**: shifts the center to 0
- **sigma (std)**: scales the spread

People tested: if we remove the mean-centering and only keep scaling, does it get worse?

Surprisingly: **not much**. Later linear layers (like W_Q, W_K, ...) can learn to handle non-zero-mean activations.

So why pay the cost of computing the mean every time? Removing it saves compute.

#### 1.3 RMSNorm: scaling only

Core formula:

```
RMS(x) = sqrt((x1^2 + x2^2 + x3^2 + x4^2) / 4)

RMSNorm(x) = x / RMS(x) * gamma
```

**Compare**:

```
LayerNorm: (x - mu) / sigma * gamma + beta   (two statistics: mu and sigma)
RMSNorm:    x / RMS(x) * gamma                (one statistic: RMS)
```

What you save:
- no mean
- no (x - mu)^2 term (you use x^2 directly)

Now compute RMSNorm on `[1, 3, 5, 7]` by hand.


In [ ]:
# === RMSNorm computed by hand ===
print("=== RMSNorm computed by hand: input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# The only statistic: RMS
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print("Step 1 - mean square = (1^2+3^2+5^2+7^2)/4")
print("       = (1+9+25+49)/4")
print(f"       = {84/4:.1f}")
print(f"  RMS = sqrt({mean_sq:.1f}) = {rms:.4f}")
print()

# RMSNorm output
x_rmsnorm = x / rms
print(f"Step 2 - RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"       x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print()
print(f"RMSNorm result: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# Verify: RMS after RMSNorm should be 1
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print()
print(f"Verify - RMS after normalization: {rms_after:.4f} (should be 1) [ok]")
print(f"       Note: the mean is not forced to 0 (mean = {x_rmsnorm.mean():.4f})")
print("       That is the key difference vs LayerNorm: no mean-centering")
print()

# Compare with LayerNorm
ln = nn.LayerNorm(4, elementwise_affine=False)
x_ln = ln(x)
print(f"LayerNorm result: {[f'{v:.4f}' for v in x_ln.tolist()]}  mean={x_ln.mean():.4f}  std={x_ln.std(unbiased=False):.4f}")
print(f"RMSNorm  result: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}  mean={x_rmsnorm.mean():.4f}  RMS={rms_after:.4f}")
print()
print("-> Different numbers, but both serve as a normalization mechanism")
print("-> LayerNorm enforces mean=0; RMSNorm does not, yet works well in practice")


In [ ]:
# === Full RMSNorm implementation ===

class RMSNorm(nn.Module):
    """RMSNorm: scale-only normalization (no mean-centering).

    Formula: y = x / RMS(x) * gamma
    RMS(x) = sqrt(mean(x^2) + eps)

    Notes:
    - No beta (bias) term in RMSNorm (often unnecessary)
    - eps prevents division by zero
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma


# Quick test
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # 2 batches, 4 tokens, 8 dims
out_rms = rmsn(batch)
out_ln = ln(batch)

print("=== RMSNorm sanity check ===")
print(f"input shape:        {batch.shape}")
print(f"RMSNorm out shape:  {out_rms.shape}")
print(f"output RMS (~1):    {torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# Parameter count comparison
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"LayerNorm params: gamma({d_model}) + beta({d_model}) = {ln_params}")
print(f"RMSNorm  params: gamma({d_model}) = {rmsn_params}")
print("-> Fewer params is not the main point; the main point is: RMSNorm skips mean computation")


---

### 2. Upgrade 2: ReLU -> SwiGLU

#### 2.1 What does ReLU do in the original FFN?

Part-4 FFN:

```
FFN(x) = W2 * ReLU(W1 * x)

ReLU: negative -> 0, positive -> unchanged
```

Let's compute it for a concrete vector.


In [ ]:
# === ReLU computed by hand ===
print("=== What ReLU does to each number ===")
print()

# Simulate the output of W1 @ x (expanded to 8 dims)
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"input (W1 @ x): {[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print("ReLU action:")
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = "keep" if h >= 0 else "-> 0 (killed)"
    print(f"  pos {i}: {h:6.2f} -> {r:6.2f}  {action}")

killed = sum(1 for h in hidden if h < 0)
print()
print(f"Issue: {killed}/8 positions are set to exactly 0")
print("      even small negatives like -0.1 or -0.01 get zeroed out")
print("      -> ReLU is a bit too brutal: all negatives become 0")


#### 2.2 Gating intuition: editor vs editor-in-chief

Imagine you write an article and submit it to an editor:

- **ReLU (one editor)**: if a paragraph is "negative" it gets deleted entirely. Even if it is 90% good and 10% flawed, it becomes 0. Wasteful.

- **Gating (two editors)**:
  - Editor A: understands content (information path)
  - Editor B: scores importance (gate path), output in [0, 1]
  - final output = A * B

So if something is "a bit wrong", B can give 0.3 and you keep part of it, instead of killing it completely.

Formula:

```
SwiGLU(x) = (W_up * x)  ⊙  Swish(W_gate * x)
              ^ info path     ^ gate path

Swish(a) = a * sigmoid(a) = a / (1 + e^{-a})
```

**What is Swish?** A smooth soft-switch, unlike ReLU's hard cutoff.


In [ ]:
# === ReLU vs Swish (SiLU) compared by hand ===
print("=== Activation functions: ReLU vs Swish ===")
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {'Note'}")
print("-" * 56)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))
    if x < 0:
        note = f"ReLU kills it, Swish keeps {swish:.4f}"
    elif x == 0:
        note = "ReLU has a kink at 0, Swish is smooth"
    else:
        note = "both pass through"
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print("Key observation:")
print("  1. Positive region: Swish ~= ReLU (slightly smaller)")
print("  2. Negative region: ReLU=0, Swish keeps a small negative value")
print("  3. At x=0: ReLU is not differentiable (mathematically), Swish is smooth")
print()
print("Why do small negative values matter?")
print("  Because the gradient through Swish in the negative region is not exactly 0")
print("  -> gradients can still flow even when a unit is weakly activated")
print("  -> ReLU units can 'die' (once negative, gradient=0 and they never recover)")


In [ ]:
# === Compute one SwiGLU step by hand ===
print("=== SwiGLU computed by hand ===")
print()

# Tiny example: d_model=4, one token
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])
d_model = x.shape[-1]

# Use tiny linear layers to make the flow visible
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

up = W_up(x)
gate = F.silu(W_gate(x))  # SiLU = Swish
gated = up * gate
out = W_down(gated)

print(f"input x: {x.tolist()}")
print()
print(f"info path (W_up @ x):        {[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"gate path Swish(W_gate @ x): {[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"gated (up * gate):           {[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"output (W_down @ gated):     {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print("Gating intuition:")
print("  If gate ~= 0 at some dimension -> that information is shut off")
print("  If gate ~= 1 -> information passes through")
print("  If gate ~= 0.3 -> only ~30% passes")


In [ ]:
# === Full SwiGLU FFN implementation ===

class FeedForward_SwiGLU(nn.Module):
    """SwiGLU FFN (used in LLaMA-like models).

    FFN(x) = W_down( Swish(W_gate(x)) * W_up(x) )

    Compared to a ReLU FFN (two matrices), SwiGLU uses three matrices.
    To keep parameter count similar, d_ff is typically reduced.
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff ~= 2 * d_model * (4 * d_model)
            # -> d_ff ~= 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # round up to a multiple of 256
            d_ff = max(d_ff, d_model)

        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))


# Parameter count comparison
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256

print("=== SwiGLU vs ReLU FFN parameter count ===")
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}x{4*d_model}) + W2({4*d_model}x{d_model})")
print(f"         = {old_p:,} parameters")
print()
print(f"SwiGLU FFN: W_gate({d_model}x{d_ff_s}) + W_up({d_model}x{d_ff_s}) + W_down({d_ff_s}x{d_model})")
print(f"         = {new_p:,} parameters")
print()
print(f"ratio: {new_p/old_p:.2f}x (close to 1 is fine)")
print("-> similar parameter count, but SwiGLU tends to work better")


---

### 3. Upgrade 3: Post-Norm -> Pre-Norm

#### 3.1 What does "Post" in Post-Norm mean?

In Part 4, the block is written like:

```python
x = x + attention(x)   # residual add
x = norm(x)            # LayerNorm
```

LayerNorm is applied **after** Attention, so it is called **Post-Norm**.

Computation graph (Post-Norm):

```
  x -> Attention -> + -> LayerNorm -> out
  |               ^
  +---------------+   (residual)
```

Notice: the residual path also goes through LayerNorm.

#### 3.2 Why is that a problem?

Residual connections are often described as "gradient highways": gradients can flow back without being weakened by Attention/FFN.

But in Post-Norm, the highway has a toll booth: **LayerNorm**.
LayerNorm rescales gradients.
In a 4-layer toy network it is fine, but in 40/80 layers the scaling can accumulate and cause exploding/vanishing gradients.

#### 3.3 Pre-Norm: move the toll booth off the highway

Pre-Norm moves normalization **before** Attention:

```python
x = x + attention(norm(x))
```

Graph (Pre-Norm):

```
  x -> LayerNorm -> Attention -> + -> out
  |                            ^
  +----------------------------+   (residual bypasses Norm)
```

Key change: the residual path bypasses normalization.
Now the gradient highway has no toll booth.

Next we compare two tiny networks.


In [ ]:
# === Post-Norm vs Pre-Norm: compare gradient flow ===
print("=== Post-Norm vs Pre-Norm: gradient flow comparison ===")
print()

# Build two deep toy nets (FFN-only) to isolate the effect of norm placement
d_model = 4
num_layers = 8

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])

    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])

    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# Same initialization
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

print(f"Depth: {num_layers} layers")
print()
print(f"{'layer':>6s}  {'Post-Norm grad':>16s}  {'Pre-Norm grad':>16s}")
print("-" * 44)

for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print("Most important: bottom layer (Layer 1) gradient")
print(f"  Post-Norm Layer 1: {grad_post_first:.6f}")
print(f"  Pre-Norm  Layer 1: {grad_pre_first:.6f}")
print(f"  Pre/Post ratio:    {grad_pre_first/grad_post_first:.2f}x")
print()
print("-> Pre-Norm usually gives larger bottom-layer gradients because the residual path bypasses the norm")
print("-> In very deep nets (e.g., 80 layers), this difference can decide whether training works")


#### 3.4 Cooking analogy summary

- **Post-Norm**: cook first, then taste. If you burn it, tasting later cannot fix it (the gradient got distorted).
- **Pre-Norm**: taste seasoning first, then cook. The signal stays stable throughout (residual path stays clean).

So for deep networks, Pre-Norm is much more stable. It tolerates larger learning rates and needs less warmup.


---

### 4. Put it together: a modern LLaMA-style block

With all three upgrades, the block becomes:

```
LLaMA Block = Pre-Norm (RMSNorm) + Attention + Pre-Norm (RMSNorm) + SwiGLU FFN

  x
  |
  +-> RMSNorm -> Attention -> +   <- residual (bypasses Norm)
  |                      |
  +----------------------+ 
  |
  +-> RMSNorm -> SwiGLU FFN -> + <- residual (bypasses Norm)
  |                         |
  +-------------------------+
  |
 out
```


In [ ]:
class LLaMABlock(nn.Module):
    """A modern LLM-style Transformer block.

    Three upgrades vs Part 4:
    1) LayerNorm -> RMSNorm
    2) Post-Norm -> Pre-Norm
    3) ReLU FFN -> SwiGLU FFN
    """
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Pre-Norm: RMSNorm before attention
        self.norm_attn = RMSNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Pre-Norm: RMSNorm before FFN
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)

    def forward(self, x, mask=None):
        # Order: norm -> sublayer -> residual
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out

        h = self.norm_ffn(x)
        x = x + self.ffn(h)

        return x

# Test
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print("=== LLaMA-style block test ===")
print(f"input shape:  {test_x.shape}")
print(f"output shape: {out_new.shape}  (same shape)")
print("But internally we upgraded everything:")
print("  - RMSNorm instead of LayerNorm")
print("  - Pre-Norm instead of Post-Norm")
print("  - SwiGLU instead of ReLU FFN")


### 5. Evolution of Transformer blocks (three generations)

```
2017  Original Transformer (Vaswani et al.)
      Post-Norm + LayerNorm + ReLU FFN
      -> runs, but hard to train deep (the paper used 6 layers)

2019  GPT-2 / GPT-3
      Pre-Norm + LayerNorm + GELU FFN
      -> norm placement fixed, can stack ~96 layers
         GELU is better than ReLU, but not the end

2023  LLaMA 2/3, DeepSeek, Qwen, Mistral, ...
      Pre-Norm + RMSNorm + SwiGLU FFN
      -> all three upgrades, faster and more stable training
```

If we rank the importance:
1. **Pre-Norm** (most important: determines whether deep training is stable)
2. **SwiGLU** (large quality gains: gating makes FFN more expressive)
3. **RMSNorm** (nice-to-have: saves compute; often smaller impact)


---

## Architecture Refinements Summary

Check these in order:

1. [ok] LayerNorm = (x - mu) / sigma * gamma + beta (two statistics)
2. [ok] RMSNorm = x / RMS(x) * gamma (one statistic; no mean-centering)
3. [ok] RMSNorm saves compute: no mu, no (x - mu)^2 (use x^2)
4. [ok] ReLU issue: negative values are killed -> gradients can cut off (dead neurons)
5. [ok] SwiGLU = info path (W_up) * gate path (Swish(W_gate)) -> selective passing
6. [ok] Swish is smooth and has non-zero gradients for negative inputs
7. [ok] Post-Norm: sublayer + residual -> Norm (residual path passes through Norm)
8. [ok] Pre-Norm: Norm -> sublayer -> + residual (residual path bypasses Norm)
9. [ok] Pre-Norm stabilizes deep training (gradient highway has no toll booth)
10. [ok] LLaMA block = Pre-Norm + RMSNorm + SwiGLU (modern standard recipe)

**One-sentence summary**: Pre-Norm solves stability (can you train?), SwiGLU improves quality (how well can you train?), RMSNorm improves efficiency (how fast?).
